::: {.sb-lesson-links}
[Exercises →](../exercises/california-housing.ipynb)
[Quiz →](../quiz/quiz.qmd)
[All files of this section (zip)](../../files/02-linear-regression.zip)
:::

::: {.sb-downloads}
[Download this notebook](../downloads/lesson-computing-metrics.ipynb){download="lesson-computing-metrics.ipynb"}  
Data: [california_housing_market.csv](../downloads/data/california_housing_market.csv){download="california_housing_market.csv"}

Put the data files in a folder named `data` next to the notebook.
:::

This notebook shows you two methods to compute the metrics used for performance evaluation and feature selection.

Along the way, a few demos 🔬 show what these statistics mean on data. The next cell loads the libraries used in the demos.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
from scipy import stats
import plotly.io as pio
import plotly.graph_objects as go
pio.templates["course"] = go.layout.Template(layout_colorway=["#3B47C4", "#E4572E", "#2E7D4F", "#F2C14E", "#8E7CC3", "#4BA3C3", "#111111", "#999999"])
pio.templates.default = "plotly_white+course"

### Fisher's F-statistic

A statistical test is a process that checks whether a hypothesis is supported or contradicted by the data at our disposal. This hypothesis, called the null hypothesis and denoted $H_{0}$, would have consequences on the properties of the observed data if it were true. These properties are summarized by a test statistic, whose value tells us how compatible the data are with $H_0$.

Fisher's F-statistic tests the following hypotheses:


* When the Fisher test is applied to the model as a whole, the null hypothesis $H_{0}$ is "the variables chosen to build the model are not jointly significant in describing the target variable". If this hypothesis is true, the F-statistic follows a Fisher distribution (F-distribution) with parameters $(p, n - p - 1)$, where $p$ is the number of explanatory variables and $n$ is the number of observations used to train the model. If the value of the F-statistic, denoted "F", falls outside the most probable regions of this distribution, we can reject the null hypothesis and conclude that the chosen model has real explanatory power on the target variable.

It may seem a little far-fetched, but all statistical tests work like that. We make an assumption; if this assumption held, the test statistic would follow a given distribution. If the actual value of the statistic lands too far from the probable values of this distribution, we are allowed to reject the null hypothesis.

Mathematically, the F-statistic is written:

$$
F = \frac{SSE / p}{SSR / (n - p - 1)}
$$



::: {.callout-tip title="Intuition"}
The F-statistic compares the part of the variation explained by the model (per variable used) with the part left unexplained (per remaining degree of freedom). If the variables were useless, they would still explain a little bit by pure chance, and F would usually be around 1. A large F means the model explains much more than chance alone could.
:::



#### See it with data 🔬: what F looks like when $H_0$ is true

Let's create 2,000 datasets where $H_0$ is true by construction: 30 observations, 3 explanatory variables, and a target made of pure random numbers, unrelated to them. For each dataset, we fit a linear regression and compute F with the formula above. If the theory is right, the 2,000 values should follow the F-distribution with parameters $(3, 30 - 3 - 1) = (3, 26)$.

In [ ]:
rng = np.random.default_rng(0)
n, p = 30, 3

def f_statistic(X, y):
    X1 = np.column_stack([np.ones(len(y)), X])
    beta, *_ = np.linalg.lstsq(X1, y, rcond=None)
    y_hat = X1 @ beta
    sse, ssr = np.sum((y_hat - y.mean()) ** 2), np.sum((y - y_hat) ** 2)
    return (sse / p) / (ssr / (n - p - 1))

f_values = np.array([f_statistic(rng.normal(size=(n, p)), rng.normal(size=n)) for _ in range(2000)])
threshold = stats.f.ppf(0.95, p, n - p - 1)
print(f"95th percentile of the simulated F values: {np.percentile(f_values, 95):.2f}")
print(f"95th percentile of the F(3, 26) distribution: {threshold:.2f}")
print(f"Share of useless models with F above the threshold: {np.mean(f_values > threshold):.1%}")

95th percentile of the simulated F values: 2.94
95th percentile of the F(3, 26) distribution: 2.98
Share of useless models with F above the threshold: 4.9%


In [ ]:
fig = px.histogram(x=f_values, nbins=60, histnorm="probability density",
                   title="F-statistics of 2,000 models trained on pure noise",
                   labels={"x": "F-statistic"})
grid = np.linspace(0, 8, 200)
fig.add_scatter(x=grid, y=stats.f.pdf(grid, p, n - p - 1), mode="lines", name="F(3, 26) density")
fig.add_vline(x=threshold, line_dash="dash", annotation_text="5% rejection zone")
fig.update_layout(yaxis_title="Density")
fig.show()

The histogram of the 2,000 simulated F values matches the theoretical F(3, 26) density, and their 95th percentile (2.94) is very close to the theoretical one (2.98). Even though none of these models has any real explanatory power, 4.9% of them land in the rejection zone: that's the 5% risk $\alpha$ we accept when we reject $H_0$. Most useless models get an F around 1 or below.

Now a model where $H_0$ is false: 30 districts of the California housing dataset, explaining the house value with the median income, the house age and the average number of rooms.

In [ ]:
housing = pd.read_csv("../data/california_housing_market.csv").sample(30, random_state=0)
F_real = f_statistic(housing[["MedInc", "HouseAge", "AveRooms"]].to_numpy(), housing["MedHouseVal"].to_numpy())
print(f"F = {F_real:.2f}   p-value = {stats.f.sf(F_real, p, n - p - 1):.2e}")

F = 15.38   p-value = 5.90e-06


With F = 15.38, this model lands far to the right of the rejection threshold (2.98). Under $H_0$, an F this large would happen with a probability of about $6 \times 10^{-6}$ (the p-value), so we reject $H_0$: these 3 variables do explain house values, even with only 30 districts.

The F-test can also compare two nested models: model 1, which includes "model_1_variables", and model 2, which includes "model_1_variables + $X_d$" ($p_1$ and $p_2$ explanatory variables respectively). In this case, the F-statistic follows an F-distribution with parameters $(p_2 - p_1, n - p_2 - 1)$ if the hypothesis that the simpler model (model 1) is enough to describe the target variable is true. The formula of F is then:

$$
F = (\frac{SSR_{1}-SSR_{2}}{p_{2}-p_{1}})(\frac{n-p_{2}-1}{SSR_{2}})
$$

If the value of the F-statistic is in an unlikely region of the F-distribution, the hypothesis is rejected, and the test suggests that the more complex model 2 brings significant additional information compared to the simpler model 1.

Graphically the F-test can be illustrated as follows:

![F-statistic](../img/f_distribution.png)

The figure shows the density of the F-distribution. As in any test, we choose a level $\alpha$ between 0 and 1, which sets the size of the rejection zone. Very often, we choose $\alpha = 5\%$ when no specific knowledge helps us adjust our standards. The F-test is one-sided: only large values of F allow us to reject the hypothesis. More precisely, if the value of F falls in the upper 5% of the expected distribution, we can say that the hypothesis is rejected at the $1 - \alpha$ = 95% confidence level.

This first metric lets us test the hypothesis that the explanatory variables have no influence on the target variable. We will now look at metrics that indicate the performance level of the model.

* $R^2_{adjusted}$

$R^2_{adjusted}$ is a modified version of $R²$ that penalizes the number of explanatory variables selected to build the model. Its mathematical formula is:

$$
R^2_{adjusted} = 1-\frac{n-1}{n-p-1}(1-R^2)
$$

where $p$ is the number of explanatory variables and $n$ is the number of observations. The growth of $R^2$ as a function of $p$ is compensated by the growth of the penalty factor $\frac{n-1}{(n-p-1)}$, which multiplies $(1-R^2)$ and increases with $p$. Consequently, if an explanatory variable does not bring enough information, $R^2_{adjusted}$ decreases. This is why we can use this indicator to compare models that do not have the same number of explanatory variables.



::: {.callout-tip title="Intuition"}
The penalty factor $\frac{n-1}{n-p-1}$ depends on how many observations you have per variable. In the small example below, with $n = 5$ observations and $p = 2$ variables, the factor is $\frac{4}{2} = 2$: the unexplained part $1 - R^2$ counts double. With the 20,640 districts of the California dataset and 8 variables, the factor is $\frac{20639}{20631} \approx 1.0004$: the adjusted $R^2$ is practically equal to $R^2$. The penalty matters most when data are scarce, which is exactly when adding variables is the most dangerous. The lesson on performance and feature selection shows this with pure-noise variables.
:::



* P-values

P-values evaluate the contribution of each explanatory variable individually, as opposed to evaluating the model as a whole. Unfortunately, they cannot easily be computed with sklearn, so we will introduce the statsmodels library, which calculates all the important metrics automatically. The p-value is the probability of observing a coefficient at least as far from 0 as the estimated one if the parameter's true value were 0, in other words if the variable did not bring any information to the linear model. A small p-value is therefore evidence that the variable is useful. Usually, we consider that a p-value below 5% means that the variable is significant, and that it is not significant otherwise. However, this depends on the context and the standards of your industry: for example, web marketing agencies typically have lower standards than pharmaceutical companies, because their goals and constraints are fundamentally different.



::: {.callout-tip title="Intuition"}
Even a completely useless variable never gets a coefficient of exactly 0, because the noise in the data always creates small accidental correlations. The p-value answers the question: "if this variable were useless, how often would chance alone produce a coefficient at least this far from 0?". If the answer is "rarely" (less than 5% of the time), we conclude that the variable is probably useful.
:::



## Computation by hand

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression

In [ ]:
# SST, SSE and SSR have to be calculated manually
# generate some example data
X = np.array([
    [1,3,5,6,7],
    [4.6, 3.7, 3.4, 3.0, 3.1]
]).transpose()
Y = np.array([2.1, 3.5, 4.4, 5.6, 5.9])

from sklearn.linear_model import LinearRegression
model = LinearRegression() # create and instanceof the model

model.fit(X,Y) # fit the model

# calculate evaluation metrics
SST = np.sum(np.square(Y - np.mean(Y)))
print("Sum of Square Total {}".format(SST))

SSE = np.sum(np.square(model.predict(X) - np.mean(Y)))
print("Sum of Square Explained {}".format(SSE))

SSR = np.sum(np.square(Y - model.predict(X)))
print("Sum of Square Residual {}".format(SSR))
print("\n")

# calculate R square and adjusted R-square
R_2 = 1 - SSR/SST
print("R square {}".format(R_2))
R_2_alt = model.score(X,Y) # alternative method to calculate R square
print("R square {}".format(R_2_alt))
n = X.shape[0]
p = X.shape[1]
R_2_adj = 1 - (n-1)/(n-p-1)*(1-R_2)
print("R square adjusted {}".format(R_2_adj))


Sum of Square Total 9.74
Sum of Square Explained 9.612353658536577
Sum of Square Residual 0.1276463414634147


R square 0.9868946261331196
R square 0.9868946261331196
R square adjusted 0.9737892522662392


## Computation with statsmodels

In [ ]:
# alternative solution with library statsmodels (useful mainly for linear models)
import statsmodels.api as sm

X2 = sm.add_constant(X) # the coefficient beta_0 also called intercept is not automatically included, so we need to manually add a constant variable equal to one.
est = sm.OLS(Y, X2)
est2 = est.fit()
print("\n")
print("-----------------------------------------------------------------------------------------")
print("------------------------Results from statsmodels-----------------------------------------")
print("-----------------------------------------------------------------------------------------")
print("\n")
print(est2.summary())



-----------------------------------------------------------------------------------------
------------------------Results from statsmodels-----------------------------------------
-----------------------------------------------------------------------------------------


                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.987
Model:                            OLS   Adj. R-squared:                  0.974
Method:                 Least Squares   F-statistic:                     75.30
Date:                Mon, 16 Nov 2026   Prob (F-statistic):             0.0131
Time:                        19:51:10   Log-Likelihood:                 2.0751
No. Observations:                   5   AIC:                             1.850
Df Residuals:                       2   BIC:                            0.6781
Df Model:                           2                                         
Covariance Type

/tmp/claude-227905993/-volatile-home-sb283337-website/1612eefd-9ad8-402e-be72-1a6f17c096ee/scratchpad/venv/lib/python3.14/site-packages/statsmodels/regression/linear_model.py:3158: ValueWarning: omni_normtest is not valid with less than 8 observations; 5 samples were given.
  omni, omnipv = cached("omni_normtest", lambda: omni_normtest(self.wresid))


## See it with data 🔬: recomputing the statsmodels results by hand

The statsmodels summary above gives many numbers. None of them is magic: let's recompute the main ones with the formulas of this lesson, using the variables of the previous cells (`SSE`, `SSR`, `n`, `p`, `X2`...).

First, the F-statistic, its p-value (the probability of getting an F at least this large under $H_0$, given by the F-distribution) and the adjusted $R^2$:

In [ ]:
F = (SSE / p) / (SSR / (n - p - 1))
print(f"F-statistic        by hand: {F:.4f}   statsmodels: {est2.fvalue:.4f}")
print(f"p-value of F       by hand: {stats.f.sf(F, p, n - p - 1):.6f}   statsmodels: {est2.f_pvalue:.6f}")
print(f"adjusted R²        by hand: {R_2_adj:.4f}   statsmodels: {est2.rsquared_adj:.4f}")

F-statistic        by hand: 75.3046   statsmodels: 75.3046
p-value of F       by hand: 0.013105   statsmodels: 0.013105
adjusted R²        by hand: 0.9738   statsmodels: 0.9738


Then the p-values of the coefficients. Each coefficient is divided by its standard error (its expected variability due to noise), which gives a t-statistic. Under the hypothesis "the true coefficient is 0", this statistic follows a Student distribution with $n - p - 1$ degrees of freedom.

In [ ]:
beta = est2.params
sigma2 = SSR / (n - p - 1)                                   # estimated variance of the noise
std_err = np.sqrt(np.diag(sigma2 * np.linalg.inv(X2.T @ X2)))
t_stat = beta / std_err
p_values = 2 * stats.t.sf(np.abs(t_stat), n - p - 1)       # two-sided test

pd.DataFrame({"coef": beta, "std err (by hand)": std_err, "std err (statsmodels)": est2.bse,
              "p-value (by hand)": p_values, "p-value (statsmodels)": est2.pvalues},
             index=["const", "x1", "x2"]).round(4)

,coef,std err (by hand),std err (statsmodels),p-value (by hand),p-value (statsmodels)
const,4.0554,3.1568,3.1568,0.3276,0.3276
x1,0.5016,0.1793,0.1793,0.1075,0.1075
x2,-0.5512,0.6719,0.6719,0.4982,0.4982


Every number matches statsmodels: F = 75.3046 with a p-value of 0.0131, an adjusted $R^2$ of 0.9738, and the same standard errors and p-values for the coefficients. Notice that with only 5 observations, the model as a whole is significant at the 5% level (p-value of F = 0.0131), but none of the coefficients is individually significant (p-values 0.3276, 0.1075 and 0.4982): with so little data, the model can't tell which of the two variables does the work.

Finally, the F-test for nested models: model 1 uses only the first variable, model 2 uses both. We apply the second formula of the lesson, and compare with statsmodels' `compare_f_test`.

In [ ]:
est_small = sm.OLS(Y, sm.add_constant(X[:, [0]])).fit()      # model 1: first variable only
SSR_1, SSR_2, p_1, p_2 = est_small.ssr, est2.ssr, 1, 2
F_nested = (SSR_1 - SSR_2) / (p_2 - p_1) * (n - p_2 - 1) / SSR_2
print(f"Nested F by hand: {F_nested:.4f}   p-value: {stats.f.sf(F_nested, p_2 - p_1, n - p_2 - 1):.4f}")
F_sm, p_sm, _ = est2.compare_f_test(est_small)
print(f"statsmodels     : {F_sm:.4f}   p-value: {p_sm:.4f}")

Nested F by hand: 0.6731   p-value: 0.4982
statsmodels     : 0.6731   p-value: 0.4982


Both computations give F = 0.6731 with a p-value of 0.4982: adding the second variable does not significantly improve the model. This p-value is exactly the p-value of `x2` in the table above. This is not a coincidence: when the two nested models differ by a single variable, the F-test is equivalent to the t-test on its coefficient ($F = t^2$, here $0.6731 \approx (-0.820)^2$).

::: {.sb-next}
**[Your turn: Linear regression exercises →](../exercises/california-housing.ipynb)**
:::